# Multi-Turn Workplace Assistant — Rollout Walkthrough

This notebook runs the multi-turn workplace-assistant benchmark end-to-end and lets you eyeball the resulting rollouts. Use it to assess whether the `user_model_system_prompt` and per-task context produce the dialogue shape you want.

**Prerequisites:**
- You've run the SDG notebook and produced `workplace_assistant_multiturn_train.jsonl`.
- The JSONL has been copied into `resources_servers/workplace_assistant/data/`.
- `env.yaml` is configured at the repo root with `policy_*` and `user_*` model settings.
- You are running this notebook from the GymMT repo root.

## 1. Point at the data

Confirm the JSONL is where we expect and inspect one row.

In [1]:
import json
from pathlib import Path

DATA_PATH = Path("/Users/artij/Projects/GymMT/resources_servers/workplace_assistant/data/workplace_assistant_multiturn_train.jsonl")
ROLLOUTS_PATH = Path("/Users/artij/Projects/GymMT/resources_servers/workplace_assistant/data/rollouts.jsonl")

assert DATA_PATH.exists(), f"missing data file: {DATA_PATH}"

with DATA_PATH.open() as f:
    rows = [json.loads(line) for line in f]

print(f"Loaded {len(rows)} tasks from {DATA_PATH}")
print("First row top-level keys:", sorted(rows[0].keys()))

Loaded 7 tasks from /Users/artij/Projects/GymMT/resources_servers/workplace_assistant/data/workplace_assistant_multiturn_train.jsonl
First row top-level keys: ['category', 'environment_name', 'ground_truth', 'id', 'responses_create_params', 'user_model_system_prompt', 'verifier_metadata']


In [2]:
# Spot-check the first task's intent + held-back info
first = rows[0]
vm = first.get("verifier_metadata", {})
print("Ambiguity type :", vm.get("ambiguity_type"))
print("Original intent:", vm.get("original_user_query"))
print("Held back      :", vm.get("removed_info"))
print("Ground truth   :", json.dumps(first.get("ground_truth"), indent=2))
print()
print("User persona prompt:")
print(first.get("user_model_system_prompt", "<missing>"))

Ambiguity type : referential_ambiguity
Original intent: Is average session duration more than 2% lower than October 3? If so, please plot it as a line chart
Held back      : The specific metric (average session duration) and the exact reference date (October 3) were replaced with vague pronouns.
Ground truth   : [
  {
    "name": "analytics_create_plot",
    "arguments": "{\"time_min\": \"2023-10-03\", \"time_max\": \"2023-11-30\", \"value_to_plot\": \"session_duration_seconds\", \"plot_type\": \"line\"}"
  }
]

User persona prompt:
                                                                                                                                                        
You are a workplace user who is utilizing an AI assistant to help you complete a task. You must stay in character as a human user and you only know what is written in your original intent below, nothing else.                                                                                                    

## 2. Start the servers

Run this in a **separate terminal** from the repo root. The command blocks and tails server logs; leave it running.

```bash
ng_run "+config_paths=[resources_servers/workplace_assistant/configs/workplace_assistant.yaml,responses_api_models/vllm_model/configs/vllm_model_for_training_with_user.yaml]"
```

Wait for `All 5 / 5 servers ready!` before moving on.

Use `vllm_model_for_training_with_user.yaml` (with `return_token_id_information: false` on both entries) if your upstream only speaks chat/completions or has Responses-API compatibility quirks. Use `openai_model_with_user.yaml` if your endpoint natively supports `/v1/responses`.

## 3. Collect rollouts

With the servers running, collect one rollout per task. Adjust `num_repeats` for variance or more samples.

In [3]:
!ng_collect_rollouts \
    +agent_name=workplace_assistant_multi_turn_agent \
    +input_jsonl_fpath=resources_servers/workplace_assistant/data/workplace_assistant_multiturn_train.jsonl \
    +output_jsonl_fpath=/tmp/wa_mt_rollouts.jsonl \
    +num_repeats=1 \
    "+responses_create_params={max_output_tokens: 8192, temperature: 0.7}"

Clearing output fpath since `resume_from_cache=False`!
Using `workplace_assistant_multi_turn_agent` for rows that do not already have an agent ref
Overriding responses_create_params fields with {'max_output_tokens': 8192, 'temperature': 0.7}
Repeating rows 1 times (in a pattern of abc to aabbcc)!
Traceback (most recent call last):
  File "/Users/artij/Projects/Gym/.venv/bin/ng_collect_rollouts", line 10, in <module>
    sys.exit(collect_rollouts())
             ^^^^^^^^^^^^^^^^^^
  File "/Users/artij/Projects/Gym/nemo_gym/rollout_collection.py", line 477, in collect_rollouts
    asyncio.run(rch.run_from_config(config))
  File "/Users/artij/.local/share/uv/python/cpython-3.12.12-macos-aarch64-none/lib/python3.12/asyncio/runners.py", line 195, in run
    return runner.run(main)
           ^^^^^^^^^^^^^^^^
  File "/Users/artij/.local/share/uv/python/cpython-3.12.12-macos-aarch64-none/lib/python3.12/asyncio/runners.py", line 118, in run
    return self._loop.run_until_complete(task)
      

## 4. Load the rollouts

In [4]:
assert ROLLOUTS_PATH.exists() and ROLLOUTS_PATH.stat().st_size > 0, (
    f"rollout file missing or empty: {ROLLOUTS_PATH}"
)

with ROLLOUTS_PATH.open() as f:
    rollouts = [json.loads(line) for line in f]

print(f"Loaded {len(rollouts)} rollouts from {ROLLOUTS_PATH}")
rewards = [r.get("reward") for r in rollouts]
print("Reward distribution:")
for r in sorted(set(rewards), key=lambda x: (x is None, x)):
    print(f"  {r!r:>8}: {rewards.count(r)}")

Loaded 7 rollouts from /Users/artij/Projects/GymMT/resources_servers/workplace_assistant/data/rollouts.jsonl
Reward distribution:
       0.0: 5
       1.0: 2


## 5. Pretty-print the dialogue

The policy trajectory lives at `response.output`. Each item is one of: `message` (system/user/assistant), `function_call`, `function_call_output`, or `reasoning`. The viewer below prints a single rollout in a readable format.

In [ ]:
def _item_text(item):
    content = item.get("content", "")
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for piece in content:
            if piece.get("type") in ("output_text", "text"):
                t = piece.get("text", "")
                if t:
                    parts.append(t)
        return "\n".join(parts)
    return ""


def format_rollout(rollout, idx, show_reasoning=False, tool_out_limit=300):
    lines = [
        "=" * 80,
        f"ROLLOUT {idx}  reward={rollout.get('reward')}",
    ]

    vm = rollout.get("verifier_metadata") or {}
    if vm.get("original_user_query"):
        lines.append(f"Original intent : {vm['original_user_query']}")
    if vm.get("ambiguity_type"):
        lines.append(f"Ambiguity type  : {vm['ambiguity_type']}")
    if vm.get("removed_info"):
        lines.append(f"Held back       : {vm['removed_info']}")
    gt = rollout.get("ground_truth") or []
    if gt:
        lines.append(f"Ground truth    : {json.dumps(gt)}")
    lines.append("-" * 80)

    output = (rollout.get("response") or {}).get("output") or []
    for item in output:
        item_type = item.get("type", "?")
        if item_type == "message":
            role = item.get("role", "").upper()
            text = _item_text(item)
            lines.append(f"[{role}]")
            for ln in (text or "").splitlines():
                lines.append("    " + ln)
        elif item_type == "function_call":
            name = item.get("name", "?")
            args = item.get("arguments", "")
            lines.append(f"[TOOL_CALL] {name}({args})")
        elif item_type == "function_call_output":
            out = item.get("output", "")
            if len(out) > tool_out_limit:
                out = out[:tool_out_limit] + f"... (+{len(out) - tool_out_limit} chars)"
            lines.append(f"[TOOL_OUT ] {out}")
        elif item_type == "reasoning":
            if show_reasoning:
                text = _item_text(item) or "(no text)"
                lines.append("[REASONING]")
                for ln in text.splitlines():
                    lines.append("    " + ln)
            # else: skip for brevity
        else:
            lines.append(f"[{item_type.upper()}] {json.dumps(item)[:200]}")

    return "\n".join(lines)


print(format_rollout(rollouts[0], 0, True))

ROLLOUT 0  reward=0.0
--------------------------------------------------------------------------------
[SYSTEM]
    Today's date is Thursday, 2023-11-30 and the current time is 23:59:00. Remember the current date and time when answering queries. Meetings must not start before 9am or end after 6pm.
[USER]
    Can you plot the distribution of both average session duration and total visits for September 13 to September 14?
[TOOL_CALL] analytics_create_plot({"time_min": "2023-09-13", "time_max": "2023-09-14", "value_to_plot": "total_visits", "plot_type": "bar"})
[TOOL_OUT ] {"output":"plots/2023-09-13_2023-09-14_total_visits_bar.png"}
[TOOL_CALL] analytics_create_plot({"time_min": "2023-09-13", "time_max": "2023-09-14", "value_to_plot": "average_session_duration", "plot_type": "bar"})
[TOOL_OUT ] {"output":"Value to plot must be one of 'total_visits', 'session_duration_seconds', 'user_engaged', 'visits_direct', 'visits_referral', 'visits_search_engine', 'visits_social_media'"}
[TOOL_CALL] 

## 6. Browse all rollouts

Print every rollout. Pass `show_reasoning=True` to include reasoning items.

In [6]:
for idx, rollout in enumerate(rollouts):
    print(format_rollout(rollout, idx))
    print()

ROLLOUT 0  reward=0.0
--------------------------------------------------------------------------------
[SYSTEM]
    Today's date is Thursday, 2023-11-30 and the current time is 23:59:00. Remember the current date and time when answering queries. Meetings must not start before 9am or end after 6pm.
[USER]
    Can you plot the distribution of both average session duration and total visits for September 13 to September 14?
[TOOL_CALL] analytics_create_plot({"time_min": "2023-09-13", "time_max": "2023-09-14", "value_to_plot": "total_visits", "plot_type": "bar"})
[TOOL_OUT ] {"output":"plots/2023-09-13_2023-09-14_total_visits_bar.png"}
[TOOL_CALL] analytics_create_plot({"time_min": "2023-09-13", "time_max": "2023-09-14", "value_to_plot": "average_session_duration", "plot_type": "bar"})
[TOOL_OUT ] {"output":"Value to plot must be one of 'total_visits', 'session_duration_seconds', 'user_engaged', 'visits_direct', 'visits_referral', 'visits_search_engine', 'visits_social_media'"}
[TOOL_CALL] 

## 7. What to look for

While reading the dialogues, flag patterns that suggest prompt tweaks:

- **User reveals too much** — leaks info the system prompt told it to hold back. Tighten the *Rules* in the converter template.
- **User reveals too little** — answers are evasive even when directly asked. Soften the "reveal only what is asked" rule.
- **Policy asks the wrong clarifying question** — check whether the held-back info is actually necessary for the ground-truth tool call.
- **Conversation terminates early** — user stop token hit or `max_turns` reached before resolution. Adjust stop token or raise `max_turns`.
- **Final tool call mismatches ground truth** — check whether the policy had the info it needed after the clarifying rounds.

When you change prompts, regenerate the JSONL in the SDG notebook, copy it back to `resources_servers/workplace_assistant/data/`, and rerun section 3.

## 8. (Optional) Reward profile summary

Once you have a larger rollout set, compute per-task pass rates.

In [ ]:
!ng_reward_profile \
    +input_jsonl_fpath=resources_servers/workplace_assistant/data/workplace_assistant_multiturn_train.jsonl \
    +rollouts_jsonl_fpath=/tmp/wa_mt_rollouts.jsonl \
    +output_jsonl_fpath=/tmp/wa_mt_profiled.jsonl \
    +pass_threshold=1.0